# Compute Bound vs Memory Bound

When optimizing code, the bottleneck is often either **how fast the CPU can calculate** or **how fast data can move through memory**. Understanding which one limits your workload guides where to spend optimization effort.

In [ ]:
## Compute bound

A workload is **compute bound** when the processor spends most of its time doing arithmetic, not waiting for data.

- The CPU is the bottleneck.
- More FLOPS (floating-point operations per second) directly improve performance.
- Typical signs: heavy math on data that fits in CPU cache, tight inner loops with few memory accesses per operation.

**Example:** computing many square roots or matrix multiplications on a small array.

In [ ]:
import time
import numpy as np

def compute_bound_workload(n: int = 1_000, iterations: int = 5_000) -> float:
    """Heavy math on a small array — data stays in cache; CPU math dominates."""
    x = np.linspace(0.1, 10.0, n, dtype=np.float64)
    start = time.perf_counter()
    for _ in range(iterations):
        x = np.sqrt(x) + np.sin(x) * np.cos(x) + np.log1p(x)
    return time.perf_counter() - start

compute_time = compute_bound_workload()
print(f"Compute-bound time: {compute_time:.3f} s")
print(f"Array size: 1,000 elements (~8 KB) — fits easily in L1/L2 cache")
print("Bottleneck: CPU arithmetic throughput, not memory bandwidth")

## Memory bound

A workload is **memory bound** when the processor spends most of its time waiting for data from RAM.

- Memory bandwidth or latency is the bottleneck.
- Faster CPUs help less; larger/faster caches and better memory access patterns matter more.
- Typical signs: simple operations on very large arrays, streaming through data once with little reuse.

**Example:** adding a constant to every element of a huge array.

In [ ]:
def memory_bound_workload(n: int = 50_000_000) -> float:
    """Simple operation on a huge array — each element is touched once; RAM bandwidth dominates."""
    x = np.ones(n, dtype=np.float64)
    start = time.perf_counter()
    x += 1.0  # one read + one write per element (~800 MB moved for 50M doubles)
    return time.perf_counter() - start

memory_time = memory_bound_workload()
bytes_moved = 50_000_000 * 8 * 2  # read + write
print(f"Memory-bound time: {memory_time:.3f} s")
print(f"Array size: 50,000,000 elements (~400 MB)")
print(f"Approx. data moved: {bytes_moved / 1e9:.2f} GB")
print(f"Effective bandwidth: {bytes_moved / memory_time / 1e9:.1f} GB/s")
print("Bottleneck: moving data between RAM and CPU, not arithmetic")

## Arithmetic intensity

**Arithmetic intensity** = operations performed ÷ bytes moved from memory.

| Intensity | Tendency | Optimize by |
|-----------|----------|-------------|
| High (many ops per byte) | Compute bound | Better algorithms, vectorization, parallelism |
| Low (few ops per byte) | Memory bound | Smaller data types, cache-friendly layout, fewer passes |

The **roofline model** plots performance vs. arithmetic intensity: low-intensity workloads hit a flat "memory roof"; high-intensity workloads hit a sloped "compute roof".

In [ ]:
def arithmetic_intensity_demo():
    """Same total data size, different ops-per-byte — shows the compute/memory trade-off."""
    n = 10_000_000
    x = np.random.rand(n)

    # Low intensity: 1 add per 8 bytes read + 8 bytes written ≈ 0.06 ops/byte
    start = time.perf_counter()
    y = x + 1.0
    low_intensity_time = time.perf_counter() - start

    # Higher intensity: reuse data in cache with chained math
    start = time.perf_counter()
    y = x
    for _ in range(50):
        y = y * 1.0001 + 0.0001
    high_intensity_time = time.perf_counter() - start

    print(f"Low intensity  (x + 1):           {low_intensity_time:.3f} s")
    print(f"Higher intensity (50 math passes): {high_intensity_time:.3f} s")
    print(f"Ratio (high / low):                {high_intensity_time / low_intensity_time:.1f}x slower")
    print()
    print("More math per memory access shifts the workload toward compute bound.")

arithmetic_intensity_demo()